# Best Experiments — DQN on ALE/Tennis-v5

It runs **10 experiments** for **Best** using the shared reusable `train.py`.


In [1]:
# Imports and setup
import os
import json
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

MEMBER = 'best'
ENV_ID = 'ALE/Tennis-v5'
SEED = 42
TOTAL_TIMESTEPS = 50_000
EVAL_EPISODES = 3

BASE_DIR = Path('results') / MEMBER
MODEL_DIR = BASE_DIR / 'models'
LOG_DIR = BASE_DIR / 'logs'
TABLE_DIR = BASE_DIR / 'tables'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print('Notebook setup complete.')
print('Base directory:', BASE_DIR)


Notebook setup complete.
Base directory: results\best


In [2]:
#  Define 10 experiment configurations
experiments = [
    {'name': 'best_exp01_baseline', 'hp': dict(learning_rate=1e-4, gamma=0.99, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=10_000, exploration_fraction=0.10, exploration_initial_eps=1.0, exploration_final_eps=0.05)},
    {'name': 'best_exp02_low_lr', 'hp': dict(learning_rate=5e-5, gamma=0.99, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=10_000, exploration_fraction=0.10, exploration_initial_eps=1.0, exploration_final_eps=0.05)},
    {'name': 'best_exp03_high_lr', 'hp': dict(learning_rate=2e-4, gamma=0.99, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=10_000, exploration_fraction=0.10, exploration_initial_eps=1.0, exploration_final_eps=0.05)},
    {'name': 'best_exp04_high_gamma', 'hp': dict(learning_rate=1e-4, gamma=0.995, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=10_000, exploration_fraction=0.10, exploration_initial_eps=1.0, exploration_final_eps=0.03)},
    {'name': 'best_exp05_very_high_gamma', 'hp': dict(learning_rate=1e-4, gamma=0.999, batch_size=40, buffer_size=60_000, learning_starts=10_000, train_freq=6, gradient_steps=2, target_update_interval=15_000, exploration_fraction=0.10, exploration_initial_eps=1.0, exploration_final_eps=0.015)},
    {'name': 'best_exp06_low_gamma_fast_updates', 'hp': dict(learning_rate=2e-4, gamma=0.97, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=2, gradient_steps=1, target_update_interval=4_000, exploration_fraction=0.10, exploration_initial_eps=1.0, exploration_final_eps=0.03)},
    {'name': 'best_exp07_large_batch', 'hp': dict(learning_rate=1e-4, gamma=0.99, batch_size=64, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=10_000, exploration_fraction=0.12, exploration_initial_eps=1.0, exploration_final_eps=0.05)},
    {'name': 'best_exp08_more_exploration', 'hp': dict(learning_rate=1e-4, gamma=0.99, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=10_000, exploration_fraction=0.20, exploration_initial_eps=1.0, exploration_final_eps=0.08)},
    {'name': 'best_exp09_less_exploration', 'hp': dict(learning_rate=1e-4, gamma=0.99, batch_size=32, buffer_size=50_000, learning_starts=10_000, train_freq=4, gradient_steps=1, target_update_interval=8_000, exploration_fraction=0.05, exploration_initial_eps=1.0, exploration_final_eps=0.02)},
    {'name': 'best_exp10_balanced_tuned', 'hp': dict(learning_rate=7.5e-5, gamma=0.995, batch_size=32, buffer_size=60_000, learning_starts=10_000, train_freq=4, gradient_steps=2, target_update_interval=8_000, exploration_fraction=0.12, exploration_initial_eps=1.0, exploration_final_eps=0.02)},
]

print(f'Prepared {len(experiments)} experiment configurations.')


Prepared 10 experiment configurations.


In [3]:
#  Helper to run one experiment through scripts/train.py
import sys
import subprocess
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR

# move up until we find scripts/train.py
while not (PROJECT_ROOT / "scripts" / "train.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_SCRIPT = PROJECT_ROOT / "scripts" / "train.py"

print("Notebook dir :", NOTEBOOK_DIR)
print("Project root :", PROJECT_ROOT)
print("Train script :", TRAIN_SCRIPT)
print("Exists?      :", TRAIN_SCRIPT.exists())

def run_experiment(exp: dict):
    hp = exp["hp"]

    cmd = [
        sys.executable, str(TRAIN_SCRIPT),
        "--member", MEMBER,
        "--experiment", exp["name"],
        "--env-id", ENV_ID,
        "--policy", "CnnPolicy",
        "--total-timesteps", str(TOTAL_TIMESTEPS),
        "--seed", str(SEED),
        "--learning-rate", str(hp["learning_rate"]),
        "--gamma", str(hp["gamma"]),
        "--batch-size", str(hp["batch_size"]),
        "--buffer-size", str(hp["buffer_size"]),
        "--learning-starts", str(hp["learning_starts"]),
        "--train-freq", str(hp["train_freq"]),
        "--gradient-steps", str(hp["gradient_steps"]),
        "--target-update-interval", str(hp["target_update_interval"]),
        "--exploration-initial-eps", str(hp["exploration_initial_eps"]),
        "--exploration-final-eps", str(hp["exploration_final_eps"]),
        "--exploration-fraction", str(hp["exploration_fraction"]),
        "--eval-freq", "10000",
        "--eval-episodes", str(EVAL_EPISODES),
    ]

    print("\nRunning:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True
    )

    print("\nSTDOUT:\n", result.stdout)
    if result.stderr:
        print("\nSTDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Experiment failed with exit code {result.returncode}")

Notebook dir : c:\Users\irabe\Downloads\Tenis\experiments
Project root : c:\Users\irabe\Downloads\Tenis
Train script : c:\Users\irabe\Downloads\Tenis\scripts\train.py
Exists?      : True


In [ ]:
#  Run all 10 experiments
for exp in experiments:
    run_experiment(exp)

print('All experiments finished.')



Running:
c:\Users\irabe\Downloads\Tenis\.venv\Scripts\python.exe c:\Users\irabe\Downloads\Tenis\scripts\train.py --member best --experiment best_exp01_baseline --env-id ALE/Tennis-v5 --policy CnnPolicy --total-timesteps 50000 --seed 42 --learning-rate 0.0001 --gamma 0.99 --batch-size 32 --buffer-size 50000 --learning-starts 10000 --train-freq 4 --gradient-steps 1 --target-update-interval 10000 --exploration-initial-eps 1.0 --exploration-final-eps 0.05 --exploration-fraction 0.1 --eval-freq 10000 --eval-episodes 3


In [ ]:
#  Read evaluation summaries into one table
rows = []
missing = []

for exp in experiments:
    eval_path = BASE_DIR / exp['name'] / f"{exp['name']}_eval.json"
    if eval_path.is_file():
        with open(eval_path, 'r', encoding='utf-8') as f:
            rows.append(json.load(f))
    else:
        missing.append(str(eval_path))

if missing:
    print('Missing evaluation files:')
    for m in missing:
        print(' -', m)

results_df = pd.DataFrame(rows).sort_values('mean_reward', ascending=False).reset_index(drop=True)
results_df


In [ ]:
#  Save results table
results_csv = TABLE_DIR / f'{MEMBER}_tennis_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'Saved results table to: {results_csv}')
results_df


In [ ]:
#  Identify Best's best model
if results_df.empty:
    raise ValueError('No results found. Run the experiments first.')

best_row = results_df.iloc[0]
best_experiment_name = best_row['experiment']
best_mean_reward = best_row['mean_reward']
best_model_path = best_row['best_model_path']

print('Best experiment:', best_experiment_name)
print('Best mean reward:', best_mean_reward)
print('Best model path:', best_model_path)


In [ ]:
#  Save best summary JSON
best_summary_path = TABLE_DIR / f'{MEMBER}_best_summary.json'
with open(best_summary_path, 'w', encoding='utf-8') as f:
    json.dump({'member': MEMBER, 'best_experiment': best_experiment_name, 'best_mean_reward': float(best_mean_reward), 'best_model_path': best_model_path}, f, indent=2)
print(f'Saved best summary to: {best_summary_path}')


In [ ]:
#  Visualization 1 — mean reward by experiment
_df = results_df.copy().sort_values('mean_reward', ascending=False)
plt.figure(figsize=(10, 4))
plt.bar(_df['experiment'], _df['mean_reward'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Mean reward (evaluation)')
plt.title('Best — DQN Tennis mean reward by experiment')
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2 — training time vs performance
plt.figure(figsize=(6, 4))
plt.scatter(results_df['train_minutes'], results_df['mean_reward'])
plt.xlabel('Train minutes')
plt.ylabel('Mean reward')
plt.title('Training time vs performance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
#  Play Best's best model
import time
import ale_py  # noqa: F401
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

def make_play_env(seed: int, render_mode='human'):
    env = make_atari_env(ENV_ID, n_envs=1, seed=seed, env_kwargs={'render_mode': render_mode})
    env = VecFrameStack(env, n_stack=4)
    return env

N_EPISODES = 1
if not os.path.isfile(best_model_path):
    print('Best model not found:', best_model_path)
else:
    env = make_play_env(SEED)
    model = DQN.load(best_model_path, env=env)
    for ep in range(N_EPISODES):
        obs = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = env.step(action)
            ep_reward += float(rewards[0])
            done = bool(dones[0])
            time.sleep(1 / 60)
        print(f'Episode {ep + 1} return: {ep_reward:.2f}')
    env.close()
